# HADO 영상 → 선수 위치 데이터 추출

사이드뷰 대회 영상에서 선수 위치(코트 좌표 m)를 프레임별로 추출합니다.

**전체 흐름**
```
영상 → 프레임 추출 → 코트 코너 4점 지정
     → YOLOv8 감지 → IoU 추적 → 호모그래피 변환
     → 코트 좌표 CSV 저장
```

> **런타임 설정**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

## 1. 환경 설치

In [ ]:
!pip install ultralytics -q

from google.colab import drive
drive.mount('/content/drive')

import cv2, csv, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from IPython.display import display, Image as IPImage
from ultralytics import YOLO

print('설치 완료 ✓')

## 2. 설정 — 영상 경로 입력

In [ ]:
# ── 여기만 수정 ─────────────────────────────────────────────
VIDEO_PATH = Path('/content/drive/MyDrive/2023 하도 챔피언스리그 스프링컵_2023.07.22_Sub_06.00.mp4')
OUTPUT_DIR = Path('/content/drive/MyDrive/HADO_extracted')

# 코트 규격 (고정)
COURT_W = 10.0   # m (가로, 팀A ↔ 팀B)
COURT_H = 6.0    # m (세로, 좌 ↔ 우)
# ────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f'영상: {VIDEO_PATH.name}')
print(f'해상도: {width}×{height}  |  FPS: {fps:.1f}  |  총 프레임: {total_frames}  |  길이: {total_frames/fps:.1f}초')

## 3. 프레임 확인

영상의 앞·중·후반 프레임을 보고 코트 코너 위치를 파악합니다.

In [ ]:
def read_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok else None

# 앞(5초) / 중간 / 뒤(75초) 프레임
sample_frames = {
    f'앞부분 ({int(fps*5)}f)':  int(fps * 5),
    f'중반 ({int(fps*40)}f)':   int(fps * 40),
    f'후반 ({int(fps*70)}f)':   int(fps * 70),
}

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, (title, fidx) in zip(axes, sample_frames.items()):
    frame = read_frame(VIDEO_PATH, fidx)
    if frame is not None:
        ax.imshow(frame)
        ax.set_title(title, fontsize=13)
        # 픽셀 좌표 격자 (100px 간격)
        for x in range(0, width, 100):
            ax.axvline(x, color='white', alpha=0.2, lw=0.5)
            if x % 200 == 0:
                ax.text(x+2, 20, str(x), color='white', fontsize=7)
        for y in range(0, height, 100):
            ax.axhline(y, color='white', alpha=0.2, lw=0.5)
            if y % 200 == 0:
                ax.text(5, y+15, str(y), color='white', fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'sample_frames.png'), dpi=100, bbox_inches='tight')
plt.show()
print('격자 간격: 100px  |  흰 숫자 = 픽셀 좌표')

## 4. 코트 코너 4점 입력

위 이미지에서 코트 바닥의 4개 코너를 찾아 픽셀 좌표를 입력합니다.

```
사이드뷰 기준 코너 배치 (카메라가 코트 옆에 있을 때):

        [먼쪽-왼쪽]─────────[먼쪽-오른쪽]
카메라    |   Team A  |  Team B   |    → 코트 위에서 본 모습
📹       [가까운-왼쪽]─────[가까운-오른쪽]

코트 좌표 매핑:
  가까운-왼쪽  = (0,   6)   Team A 가까운 코너
  가까운-오른쪽 = (10,  6)   Team B 가까운 코너
  먼쪽-왼쪽   = (0,   0)   Team A 먼 코너
  먼쪽-오른쪽  = (10,  0)   Team B 먼 코너
```

> **팁**: 코트 라인(흰 선)이 바닥과 만나는 모서리가 기준점입니다.

In [ ]:
# ── 코너 픽셀 좌표 직접 입력 (x, y) ───────────────────────
# 위 이미지의 흰색 격자 숫자를 보고 입력하세요
corners_pixel = np.array([
    [  0,   0],   # ① 가까운-왼쪽  (Team A, 카메라 쪽)
    [  0,   0],   # ② 가까운-오른쪽 (Team B, 카메라 쪽)
    [  0,   0],   # ③ 먼쪽-왼쪽   (Team A, 먼 쪽)
    [  0,   0],   # ④ 먼쪽-오른쪽  (Team B, 먼 쪽)
], dtype=np.float32)
# ────────────────────────────────────────────────────────────

# 대응하는 코트 좌표 (고정, 수정 불필요)
corners_court = np.array([
    [ 0.0, 6.0],   # ① 가까운-왼쪽
    [10.0, 6.0],   # ② 가까운-오른쪽
    [ 0.0, 0.0],   # ③ 먼쪽-왼쪽
    [10.0, 0.0],   # ④ 먼쪽-오른쪽
], dtype=np.float32)

# 호모그래피 계산
H, _ = cv2.findHomography(corners_pixel, corners_court)
H_inv, _ = cv2.findHomography(corners_court, corners_pixel)
print('호모그래피 계산 완료')

# 검증: 4개 코너를 변환해서 코트 좌표가 맞는지 확인
print('\n코너 변환 검증:')
labels = ['가까운-왼쪽', '가까운-오른쪽', '먼쪽-왼쪽', '먼쪽-오른쪽']
for label, px, ct in zip(labels, corners_pixel, corners_court):
    px_h = np.array([[[px[0], px[1]]]], dtype=np.float32)
    result = cv2.perspectiveTransform(px_h, H)[0][0]
    err = np.linalg.norm(result - ct)
    status = '✓' if err < 0.01 else '✗'
    print(f'  {status} {label}: 픽셀{tuple(px.astype(int))} → 코트({result[0]:.2f}, {result[1]:.2f}) m  [오차 {err:.4f}m]')

## 5. 호모그래피 시각화 확인

코너 지정이 올바른지 프레임에 오버레이해서 확인합니다.

In [ ]:
frame = read_frame(VIDEO_PATH, int(fps * 5))
vis = frame.copy()

# 코트 코너 표시
colors_vis = [(255,80,80), (80,255,80), (80,80,255), (255,255,80)]
for i, (px, label) in enumerate(zip(corners_pixel.astype(int), labels)):
    cv2.circle(vis, tuple(px), 10, colors_vis[i], -1)
    cv2.putText(vis, f'{i+1}:{label}', (px[0]+12, px[1]-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, colors_vis[i], 2)

# 코트 외곽선 투영 (픽셀 공간)
court_outline = np.array([
    [[0,0]], [[10,0]], [[10,6]], [[0,6]]
], dtype=np.float32)
outline_px = cv2.perspectiveTransform(court_outline, H_inv).astype(np.int32)
cv2.polylines(vis, [outline_px.reshape(-1,1,2)], True, (255,200,0), 2)

# 중앙선 (x=5.0m)
mid_pts = np.array([[[5.0,0.0]], [[5.0,6.0]]], dtype=np.float32)
mid_px  = cv2.perspectiveTransform(mid_pts, H_inv).astype(np.int32)
cv2.line(vis, tuple(mid_px[0][0]), tuple(mid_px[1][0]), (180,180,60), 2)

plt.figure(figsize=(14,8))
plt.imshow(vis)
plt.title('호모그래피 검증 — 노란 선이 코트 외곽, 연두 선이 중앙선')
plt.axis('off')
plt.savefig(str(OUTPUT_DIR / 'homography_check.png'), dpi=100, bbox_inches='tight')
plt.show()
print('코트 외곽선이 실제 코트 라인과 일치하면 다음 단계로 진행하세요.')

## 6. YOLOv8 + IoU 추적기 정의

In [ ]:
model = YOLO('yolov8n.pt')
print(f'YOLOv8n 로드 완료 | 디바이스: {next(model.model.parameters()).device}')

# ── 간단한 IoU 추적기 ─────────────────────────────────────
class SimpleTracker:
    def __init__(self, iou_thr=0.3, max_lost=15):
        self.tracks = {}   # id → {'bbox': [x1,y1,x2,y2], 'lost': 0}
        self.next_id = 1
        self.iou_thr = iou_thr
        self.max_lost = max_lost

    def _iou(self, a, b):
        xi1, yi1 = max(a[0],b[0]), max(a[1],b[1])
        xi2, yi2 = min(a[2],b[2]), min(a[3],b[3])
        inter = max(0, xi2-xi1) * max(0, yi2-yi1)
        ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
        return inter / ua if ua > 0 else 0

    def update(self, detections):
        """detections: list of [x1,y1,x2,y2,conf]"""
        matched_det = set()
        matched_trk = set()

        track_ids = list(self.tracks.keys())
        for di, det in enumerate(detections):
            best_iou, best_tid = 0, None
            for tid in track_ids:
                if tid in matched_trk: continue
                iou = self._iou(det[:4], self.tracks[tid]['bbox'])
                if iou > best_iou:
                    best_iou, best_tid = iou, tid
            if best_iou >= self.iou_thr:
                self.tracks[best_tid]['bbox'] = det[:4]
                self.tracks[best_tid]['lost'] = 0
                self.tracks[best_tid]['conf'] = det[4]
                matched_det.add(di)
                matched_trk.add(best_tid)

        # 신규 트랙
        for di, det in enumerate(detections):
            if di not in matched_det:
                self.tracks[self.next_id] = {'bbox': det[:4], 'lost': 0, 'conf': det[4]}
                self.next_id += 1

        # lost 증가 / 삭제
        for tid in list(self.tracks):
            if tid not in matched_trk:
                self.tracks[tid]['lost'] += 1
                if self.tracks[tid]['lost'] > self.max_lost:
                    del self.tracks[tid]

        return [(tid, t['bbox'], t['conf'])
                for tid, t in self.tracks.items() if t['lost'] == 0]

def bbox_to_foot(bbox):
    """바운딩 박스 → 발 위치 픽셀 (하단 중앙)"""
    return [(bbox[0]+bbox[2])/2, bbox[3]]

def pixel_to_court(px, py):
    """픽셀 좌표 → 코트 좌표 (m)"""
    pt = np.array([[[px, py]]], dtype=np.float32)
    result = cv2.perspectiveTransform(pt, H)[0][0]
    return float(result[0]), float(result[1])

print('추적기 준비 완료')

## 7. 전체 영상 처리 → CSV 추출

In [ ]:
from tqdm.notebook import tqdm

PROCESS_EVERY_N = 3   # 매 3프레임마다 처리 (속도↑, 8fps 효과)
CONF_THR = 0.35

output_csv = OUTPUT_DIR / (VIDEO_PATH.stem + '_positions.csv')
CSV_HEADER = ['frame_idx','time_sec','track_id','team','court_x','court_y',
              'pixel_x','pixel_y','confidence']

tracker = SimpleTracker(iou_thr=0.3, max_lost=15)
cap     = cv2.VideoCapture(str(VIDEO_PATH))
rows    = []

for frame_idx in tqdm(range(total_frames), desc='프레임 처리'):
    ok, frame = cap.read()
    if not ok:
        break

    if frame_idx % PROCESS_EVERY_N != 0:
        continue

    results = model(frame, classes=[0], conf=CONF_THR,
                    imgsz=640, verbose=False)[0]
    detections = []
    if results.boxes is not None:
        for box in results.boxes:
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            detections.append([x1,y1,x2,y2,conf])

    tracks = tracker.update(detections)
    time_sec = frame_idx / fps

    for tid, bbox, conf in tracks:
        fx, fy = bbox_to_foot(bbox)
        cx, cy = pixel_to_court(fx, fy)
        # 코트 범위 밖 좌표 필터링
        if not (0 <= cx <= COURT_W and 0 <= cy <= COURT_H):
            continue
        team = 'A' if cx < COURT_W/2 else 'B'
        rows.append([frame_idx, round(time_sec,3), tid, team,
                     round(cx,3), round(cy,3),
                     round(fx,1), round(fy,1), round(conf,3)])

cap.release()

# CSV 저장
with open(output_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(CSV_HEADER)
    writer.writerows(rows)

print(f'\n완료: {len(rows)}개 행 → {output_csv.name}')
print(f'처리 프레임: {total_frames // PROCESS_EVERY_N}개')
print(f'등장 ID 수: {len(set(r[2] for r in rows))}')

## 8. 결과 시각화 — 버드아이뷰 궤적

In [ ]:
import pandas as pd

df = pd.read_csv(output_csv)
print(df.head(10).to_string())
print(f'\n팀별 감지 수: {df.groupby("team")["frame_idx"].count().to_dict()}')
print(f'ID별 등장 프레임:\n{df.groupby("track_id")["frame_idx"].count().sort_values(ascending=False)}')

In [ ]:
# 선수별 이동 궤적 버드아이뷰
COLORS = ['#FF6B35','#FFB347','#FFF176',  # Team A
          '#4FC3F7','#29B6F6','#81D4FA']   # Team B

fig, ax = plt.subplots(figsize=(14, 8))

# 코트 배경
ax.set_facecolor('#1a2a1a')
ax.set_xlim(-0.2, COURT_W+0.2)
ax.set_ylim(-0.2, COURT_H+0.2)
ax.set_xlabel('코트 X (m) — Team A ← 0 ~~ 10 → Team B', fontsize=11)
ax.set_ylabel('코트 Y (m)', fontsize=11)
ax.set_title(f'선수 궤적 — {VIDEO_PATH.stem}', fontsize=13)

# 그리드
for x in np.arange(0, COURT_W+1, 1):
    ax.axvline(x, color='#333', lw=0.5)
for y in np.arange(0, COURT_H+1, 1):
    ax.axhline(y, color='#333', lw=0.5)

# 구역선
for xm in [1.5, 3.0, 7.0, 8.5]:
    ax.axvline(xm, color='#4a8', lw=1, ls='--', alpha=0.7)
# 레인선
for ym in [2.0, 4.0]:
    ax.axhline(ym, color='#668', lw=1, ls='--', alpha=0.7)
# 중앙선
ax.axvline(5.0, color='#aa8', lw=2)
ax.add_patch(patches.Rectangle((0,0), COURT_W, COURT_H,
                                 fill=False, edgecolor='white', lw=2))

# 등장 프레임 많은 상위 6개 ID만 표시
top_ids = df.groupby('track_id')['frame_idx'].count().nlargest(6).index.tolist()

for i, tid in enumerate(top_ids):
    sub = df[df['track_id'] == tid].sort_values('frame_idx')
    color = COLORS[i % len(COLORS)]
    team  = sub['team'].mode()[0]
    ax.plot(sub['court_x'], sub['court_y'],
            color=color, alpha=0.5, lw=1)
    ax.scatter(sub['court_x'].iloc[-1], sub['court_y'].iloc[-1],
               color=color, s=80, zorder=5,
               label=f'ID {tid} (팀{team})')

ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'trajectory.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'저장: {OUTPUT_DIR}/trajectory.png')

## 9. 품질 확인 체크리스트

결과를 보고 아래 항목을 확인하세요:

| 확인 항목 | 기준 |
|-----------|------|
| 궤적이 코트 안에 있는가 | 대부분 x:0~10, y:0~6 범위 |
| 선수 6명이 구분되는가 | 상위 6개 ID가 팀A/B 각 3명 |
| 궤적이 자기 진영 안에 있는가 | 팀A x<5, 팀B x>5 |
| 튀는 좌표가 없는가 | 갑자기 코트 반대편으로 점프 없음 |

**문제가 있으면** → 셀 4로 돌아가 코너 좌표를 수정 후 셀 5~8 재실행